# Notebook 094: Databricks Asset Bundles (DABs) for Multi-Environment CI/CD Automation

**Module:** Module 09: Databricks Platform, Unity Catalog, Security & FinOps  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 21: CICD process in Databricks with Bundles | Demo by Sr Data Architect  

---

## 1. Business Scenario & Real-World Context
An enterprise development team deploys 50 Databricks workflows and pipelines across development, staging, and production workspaces. Manual UI deployments cause configuration drift, broken paths, and deployment bugs.

## 2. Technical Requirement & Problem Statement
Master Databricks Asset Bundles (DABs): configure `databricks.yml`, define multi-environment deployment targets (dev, staging, prod), configure parameterized workflows, and automate CI/CD validation (`bundle validate`, `bundle deploy`, `bundle run`).

## 3. Spark & Databricks Architecture Notes
Databricks Asset Bundles (DABs) provide Infrastructure-as-Code (IaC) for Databricks projects. A YAML bundle definition (`databricks.yml`) manages notebooks, pipelines, jobs, and permissions, allowing automated deployments via GitHub Actions or GitLab CI.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_094_Databricks_Asset_Bundles_(DABs)_for_Multi-Environment_CI/CD_Automation")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Designing databricks.yml Bundle Configuration
Create production-ready YAML specification for multi-environment deployment.

In [ ]:
bundle_yaml_content = """
bundle:
  name: enterprise_spark_masterclass

include:
  - resources/*.yml

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: https://adb-dev.azuredatabricks.net
      root_path: /Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}

  staging:
    mode: production
    workspace:
      host: https://adb-staging.azuredatabricks.net
      root_path: /Shared/.bundle/${bundle.name}/${bundle.target}

  prod:
    mode: production
    workspace:
      host: https://adb-prod.azuredatabricks.net
      root_path: /Shared/.bundle/${bundle.name}/${bundle.target}
"""

print("databricks.yml configuration structure:")
print(bundle_yaml_content)

## 6. Defining Workflow Job Resource in Bundle
Declare jobs, schedules, and cluster settings in resources/jobs.yml.

In [ ]:
job_resource_yaml = """
resources:
  jobs:
    nightly_etl_pipeline:
      name: "[${bundle.target}] Nightly Financial ETL"
      tasks:
        - task_key: run_ingestion
          job_cluster_key: etl_cluster
          notebook_task:
            notebook_path: ../notebooks/01_ingest.py
        - task_key: run_transformation
          depends_on:
            - task_key: run_ingestion
          job_cluster_key: etl_cluster
          notebook_task:
            notebook_path: ../notebooks/02_transform.py
      job_clusters:
        - job_cluster_key: etl_cluster
          new_cluster:
            spark_version: 14.3.x-scala2.12
            node_type_id: Standard_D4ds_v5
            num_workers: 2
"""

print("resources/jobs.yml definition structure:")
print(job_resource_yaml)

## 7. DABs CLI Command Lifecycle Cheatsheet
Review developer CLI workflows for DABs.

In [ ]:
commands = [
    ("databricks bundle init", "Initializes a new bundle template project."),
    ("databricks bundle validate", "Validates YAML syntax, resource references, and paths."),
    ("databricks bundle deploy -t dev", "Deploys resources to target environment (dev sandbox)."),
    ("databricks bundle run -t dev nightly_etl_pipeline", "Triggers immediate run of the deployed workflow in dev.")
]

for cmd, desc in commands:
    print(f" $ {cmd}\n   -> {desc}")

## 9. Key Architectural Takeaways
- **Databricks Asset Bundles (DABs) implement Infrastructure-as-Code (IaC) for Databricks projects.**
- **Target environments (dev, staging, prod) enable seamless promotion via GitHub Actions CI/CD.**
- **DABs eliminate manual workspace UI changes, preventing configuration drift.**

## 10. Senior Data Engineering Interview Questions
1. What is a Databricks Asset Bundle (DAB) and what problem does it solve in enterprise MLOps/DataOps?
2. What is the difference between `mode: development` and `mode: production` in `databricks.yml`?
3. How do you parameterize cluster configurations across dev and prod targets in DABs?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
